# face-cloak: run one experiment on Google Colab

This notebook is a thin wrapper around the `facecloak` package in this repo. All logic lives in `src/facecloak/`,
so the result is identical to running `python scripts/run_experiment.py --config ...` on a workstation.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Outputs go to Google Drive under `MyDrive/CRPII/exp/` (shared with the team):

```
exp/_shared/images/<id>/clean.png       preprocessed inputs (shared by all experiments)
exp/_shared/masks/                       MediaPipe face masks
exp/_shared/inpaint_<hash>/              clean / random-branch inpaintings (reused by all experiments)
exp/<EXP>/cloak_out/                     manifest.csv, cloaks, figures
exp/<EXP>/inpaint_out/                   generations.csv, results.csv, summary.csv, paired.csv, figures
exp/tau.json                             fixed identity threshold (Step 0)
```

Every stage resumes from what is already on Drive, so a disconnect only costs the batch in progress.

In [ ]:
!nvidia-smi -L

## 1. Mount Drive, get the code, install

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
REPO_URL = "https://github.com/dinhieufam/face-cloak.git"
REPO_DIR = "/content/face-cloak"
BRANCH   = "main"

import os
if not os.path.exists(REPO_DIR):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
!git -C {REPO_DIR} log -1 --oneline

In [ ]:
# A CPU-only onnxruntime would shadow the GPU build that uniface[gpu] installs, so remove it first.
!pip uninstall -y -q onnxruntime
!pip install -q -e "{REPO_DIR}[eval]"

import sys
sys.path.insert(0, f"{REPO_DIR}/src")   # editable installs are only picked up after a restart otherwise

## 2. Pick the experiment

| EXP | change vs. baseline |
|---|---|
| `E0_baseline` | none: PhotoGuard PGD, gray target (also runs the random-noise control) |
| `E1_pattern_target` | target image = `assets/targets/pattern.png` |
| `E2_mask_aware` | loss on the attacker's view (face on gray) + mask jitter |
| `E3_momentum` | MI-FGSM, momentum 0.9 |
| `E4_untargeted` | push away from E(x) instead of pulling to a target |

`OVERRIDES` takes dotted keys, e.g. `["quick_test=true"]` (2 images, 50 iterations, ~3 min) or `["cloak.eps=[8,16]"]`.
See `configs/base.yaml` for every key and `docs/experiments.md` for the protocol.

In [ ]:
EXP       = "E0_baseline"
OVERRIDES = []

from facecloak import env
from facecloak.config import get_paths, load_config
from facecloak.viz import show

cfg   = load_config(f"{REPO_DIR}/configs/experiments/{EXP}.yaml", OVERRIDES)
paths = get_paths(cfg)
print(env.describe())
print("experiment:", cfg["exp_name"], "->", paths.exp_dir)

### (optional, once) Reuse the clean images from the old notebooks

The legacy notebooks stored inputs in `MyDrive/CRPII/cloak_out/images/<id>/clean.png`. Copying them avoids
re-downloading from Kaggle. If Kaggle asks for credentials, add `KAGGLE_USERNAME` and `KAGGLE_KEY` in the
🔑 Secrets panel (see `docs/setup.md`).

In [ ]:
import shutil, glob
LEGACY = "/content/drive/MyDrive/CRPII/cloak_out/images"
n = 0
for src in glob.glob(f"{LEGACY}/*/clean.png"):
    dst = paths.shared_images / os.path.basename(os.path.dirname(src)) / "clean.png"
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy(src, dst); n += 1
print(f"copied {n} clean images into {paths.shared_images}")

## 3. Step 0: fixed identity threshold τ (once for the whole team)

Skipped automatically when `exp/tau.json` already exists. ~300 FFHQ faces, ArcFace only (no diffusion).

In [ ]:
if paths.tau_file.exists():
    print(paths.tau_file.read_text()[:400])
else:
    !cd {REPO_DIR} && python scripts/calibrate_tau.py --config configs/base.yaml

## 4. Stage 1: cloak (PGD on the SD VAE encoder)

In [ ]:
from facecloak.cloak import run_cloak, summarize_cloak

manifest = run_cloak(cfg)
display(summarize_cloak(manifest))
for f in sorted((paths.cloak_dir / "figures").glob("*.png")):
    show(f)

## 5. Stage 2: inpainting attack (SD 1.5)

In [ ]:
from facecloak.inpaint import run_inpaint

gens = run_inpaint(cfg)

## 6. Stage 3: ArcFace evaluation

In [ ]:
from facecloak.evaluate import run_eval

out = run_eval(cfg)
print(f"τ = {out['tau']:.3f}  ({out['tau_info']['tau_source']})")
display(out["summary"])
display(out["paired"])
for f in sorted((paths.inpaint_dir / "figures").glob("*.png")):
    show(f)

## 7. Compare all finished experiments

In [ ]:
from facecloak.evaluate import compare_experiments

table = compare_experiments(paths.root)
table.to_csv(paths.root / "comparison.csv", index=False)
display(table)